# model_experiment_DLinear

DLinear (Zeng et al. 2022, *"Are Transformers Effective for Time Series Forecasting?"*) — implemented from the paper in `src/dlinear.py` (~50 lines of PyTorch, no forecasting library).

Architecture: decompose each input window into **trend** (moving average) and **seasonal remainder**, apply one linear layer to each, sum. Channel-independent: one shared model, each of the 3331 series is forecast from its own history in a single shot — direct multi-horizon over all 39 test weeks, no recursion.

Training: per-window standardization (sales scales differ 100x across series), weighted L1 loss with `1 + 4*IsHoliday` on target weeks — the exact competition metric. Series without enough history fall back to seasonal naive.

MLflow experiment: **DLinear_Training** — runs: Preprocessing, CV_input*, Fold2, Blend_*, Final. CPU is enough (the model has ~8K parameters).

In [ ]:
# Kaggle bootstrap — does nothing when running locally.
# On Kaggle: attach the competition data (Add Input) and the three
# MLFLOW_TRACKING_* secrets (Add-ons -> Secrets) before Run All.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path (local runs)

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow
from src.dlinear import build_wide, train_dlinear, forecast
from src.postprocess import naive_lag52, apply_christmas_shift, blend_holiday_naive
from src.preprocessing import BLEND_HOLIDAY_WEEKS

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
print(train.shape, test.shape)

setup_mlflow("DLinear_Training")

## Run 1 — Preprocessing decisions

In [ ]:
with mlflow.start_run(run_name="DLinear_Preprocessing"):
    mlflow.log_params({
        "series_format": "regular weekly grid per (Store, Dept); gaps inside the active range -> 0",
        "normalization": "per-window standardization, std floored at $1",
        "loss": "L1 weighted 1+4*IsHoliday on target weeks (matches WMAE)",
        "fallback": "seasonal naive (lag 52/53/51) + dept median for series without a full input window",
        "horizon": 39,
    })

## Run 2 — Input-window ablation on Fold 1

Note the data constraint: Fold-1 train has only 91 weeks, so input 52 + horizon 39 leaves exactly ONE training window per series; input 26 gives 27 windows per series. A trade-off between window length and training data — worth a paragraph in the README.

In [ ]:
def eval_fold(fold, input_size, epochs=30):
    tr, va = split_fold(train, fold)
    wide_tr = build_wide(tr)
    val_dates = sorted(va["Date"].unique())
    model = train_dlinear(wide_tr, input_size, len(val_dates), epochs=epochs, verbose=False)
    fc = forecast(model, wide_tr, input_size, len(val_dates), val_dates)
    m = va.merge(fc, on=["Store", "Dept", "Date"], how="left")
    coverage = m["pred"].notna().mean()
    m["pred"] = m["pred"].fillna(pd.Series(naive_lag52(tr, va), index=m.index))
    dep_med = tr.groupby("Dept")["Weekly_Sales"].median()
    m["pred"] = m["pred"].fillna(m["Dept"].map(dep_med)).fillna(0)
    rep = wmae_report(m["Weekly_Sales"], m["pred"], m["IsHoliday"])
    return rep, coverage, m, tr, model

results = {}
for L in (26, 39, 52):
    rep, cov, _, _, _ = eval_fold(1, L)
    results[L] = rep["wmae"]
    with mlflow.start_run(run_name=f"DLinear_CV_input{L}"):
        mlflow.log_params({"input_size": L, "fold": 1, "epochs": 30})
        mlflow.log_metrics({**rep, "coverage": cov})
    print(f"input={L}: coverage {cov:.3f}, {rep}")

BEST_INPUT = min(results, key=results.get)
print("best input size:", BEST_INPUT)

## Run 3 — Fold 2 sanity check

In [ ]:
rep2, cov2, _, _, _ = eval_fold(2, BEST_INPUT)
with mlflow.start_run(run_name="DLinear_Fold2"):
    mlflow.log_params({"input_size": BEST_INPUT, "fold": 2})
    mlflow.log_metrics({**rep2, "coverage": cov2})
print(rep2)

## Run 4 — Holiday blend (no Christmas — see LightGBM notebook for why)

In [ ]:
rep1, _, m1, tr1, _ = eval_fold(1, BEST_INPUT)
blend_scores = {}
for w in (0.0, 0.5, 0.75):
    blended = blend_holiday_naive(m1[["Store", "Dept", "Date", "pred"]], tr1,
                                  weight=w, holiday_dates=BLEND_HOLIDAY_WEEKS)
    rep = wmae_report(m1["Weekly_Sales"], blended["pred"], m1["IsHoliday"])
    blend_scores[w] = rep["wmae"]
    with mlflow.start_run(run_name=f"DLinear_Blend_noXmas_w{w}"):
        mlflow.log_params({"input_size": BEST_INPUT, "blend_weight": w, "fold": 1})
        mlflow.log_metrics(rep)
    print(f"w={w}: {rep}")
BLEND_W = min(blend_scores, key=blend_scores.get)
print("best blend weight:", BLEND_W)

## Run 5 — Final: train on ALL data, predict test, submission

In [ ]:
wide_full = build_wide(train)
test_dates = sorted(test["Date"].unique())
final_model = train_dlinear(wide_full, BEST_INPUT, len(test_dates), epochs=30)

fc = forecast(final_model, wide_full, BEST_INPUT, len(test_dates), test_dates)
sub = test[["Store", "Dept", "Date"]].merge(fc, on=["Store", "Dept", "Date"], how="left")
covered = sub["pred"].notna().mean()
sub["pred"] = sub["pred"].fillna(pd.Series(naive_lag52(train, test), index=sub.index))
dep_med = train.groupby("Dept")["Weekly_Sales"].median()
sub["pred"] = sub["pred"].fillna(sub["Dept"].map(dep_med)).fillna(0)

sub = apply_christmas_shift(sub, pred_col="pred")
if BLEND_W > 0:
    sub = blend_holiday_naive(sub, train, weight=BLEND_W, holiday_dates=BLEND_HOLIDAY_WEEKS)

with mlflow.start_run(run_name="DLinear_Final"):
    mlflow.log_params({"input_size": BEST_INPUT, "epochs": 30, "blend_weight": BLEND_W,
                       "post": "christmas_shift + noXmas_blend"})
    mlflow.log_metrics({"fold1_wmae": results[BEST_INPUT], "test_coverage": covered})
    # input_example batch of 2: the pt2 export marks the batch dim dynamic,
    # and a batch-of-1 example makes torch specialize it -> export error
    mlflow.pytorch.log_model(final_model, name="model",
                             input_example=np.zeros((2, BEST_INPUT), dtype=np.float32),
                             registered_model_name="walmart-dlinear")

make_submission(sub, "pred", "submission_dlinear.csv")
print(f"wrote submission_dlinear.csv (coverage {covered:.3f}, blend w={BLEND_W})")